# SQL 01. NULL・JOIN・窓関数

DuckDB を使う。サーバも設定も要らず、CSV も pandas の DataFrame も
そのままテーブルとして扱える。

**pandas と結果が食い違うところ**を中心に集めてある。同じつもりで書いて
数字が変わるのは、たいてい NULL の扱いか JOIN の行数。

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.width", 200)

con = duckdb.connect()

def q(sql):
    """SQLを実行して DataFrame で返す。"""
    return con.sql(sql).df()

# CSVから型を付けてテーブルを作る。
#   all_varchar=true … 一旦すべて文字列で読む (推測させない)
#   nullstr          … 欠損表現
#   nullif(x, '')    … 空文字も欠損にする
con.execute("""
CREATE TABLE orders AS
SELECT order_id,
       nullif(customer_id, '')                        AS customer_id,
       TRY_CAST(nullif(order_date, '') AS DATE)       AS order_date,
       lower(region)                                  AS region,
       status,
       TRY_CAST(qty AS INTEGER)                       AS qty,
       TRY_CAST(replace(amount, ',', '') AS BIGINT)   AS amount
FROM read_csv('/data/orders.csv', all_varchar=true, nullstr=['N/A', '-', 'NULL']);

CREATE TABLE customers AS
SELECT customer_id, name, nullif(tier, '') AS tier,
       TRY_CAST(signup_date AS DATE) AS signup_date
FROM read_csv('/data/customers.csv', all_varchar=true);

CREATE TABLE payments AS
SELECT payment_id, order_id, TRY_CAST(paid_at AS DATE) AS paid_at,
       TRY_CAST(amount AS BIGINT) AS amount
FROM read_csv('/data/payments.csv', all_varchar=true);
""")

q("SELECT * FROM orders")

In [ ]:
# 他の2つのテーブルも見ておく
display(q("SELECT * FROM customers"))
q("SELECT * FROM payments")

`orders.customer_id` には `C-99`(customers に無い)と欠損が1件ずつある。
`payments` には `O-001` に対する支払いが2件、`O-999`(orders に無い)が1件ある。
JOIN の練習用にわざとそうしてある。

---
## 1. NULL の三値論理

### 1-1. = NULL は成立しない

NULL は「値が無い」ではなく「値が不明」。不明どうしを比べても不明。

**実行する前に、違いを予想する。**

In [ ]:
# A: = NULL
q('''
SELECT count(*) AS n FROM orders WHERE customer_id = NULL
''')

In [ ]:
# B: IS NULL
q('''
SELECT count(*) AS n FROM orders WHERE customer_id IS NULL
''')

<details>
<summary>何が起きたか</summary>

`A` は **0件**。エラーにはならない。

`customer_id = NULL` は `TRUE` でも `FALSE` でもなく **`NULL`(不明)** を返す。
`WHERE` は `TRUE` の行だけ通すので、1行も通らない。

これが SQL の**三値論理**。`TRUE` / `FALSE` / `NULL` の3つがある。

| 式 | 結果 |
| --- | --- |
| `NULL = NULL` | `NULL` |
| `NULL <> NULL` | `NULL` |
| `NULL IS NULL` | `TRUE` |
| `NULL IS NOT DISTINCT FROM NULL` | `TRUE` |

比較には `IS NULL` / `IS NOT NULL` を使う。
「NULL も含めて等しいか」を比べたいなら `IS NOT DISTINCT FROM`。

</details>

### 1-2. NOT IN と NULL

サブクエリに NULL が1つでも入ると、`NOT IN` は何も返さなくなる。

**実行する前に、違いを予想する。**

In [ ]:
# A: NOT IN
q('''
SELECT count(*) AS n
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders)
''')

In [ ]:
# B: NOT EXISTS
q('''
SELECT count(*) AS n
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
''')

<details>
<summary>何が起きたか</summary>

`A` は **0件**。`B` は **1件**(注文の無い `C-05`)。`B` が正しい。

`orders.customer_id` には NULL が1件ある。すると

```
'C-05' NOT IN ('C-01', ..., NULL)
  = 'C-05' <> 'C-01' AND ... AND 'C-05' <> NULL
  = TRUE AND ... AND NULL
  = NULL          ← TRUE にならないので通らない
```

**サブクエリに NULL が1つでもあれば `NOT IN` は必ず0件になる。**
エラーも警告も出ない。

対策は3つ。

- `NOT EXISTS` を使う(**これが基本**)
- `LEFT JOIN ... WHERE 右.key IS NULL`(anti join)
- サブクエリ側で `WHERE key IS NOT NULL` を付ける

`IN`(否定でないほう)は NULL があっても壊れないので、余計に気づきにくい。

</details>

### 1-3. COUNT(*) と COUNT(col)

件数の数え方。pandas の `size` と `count` に対応する。

**実行する前に、違いを予想する。**

In [ ]:
# A: 3つ並べる
q('''
SELECT count(*)                 AS c_star,
       count(amount)            AS c_amount,
       count(DISTINCT region)   AS c_region,
       sum(amount)              AS total,
       avg(amount)              AS average
FROM orders
''')

In [ ]:
# B: 自分で計算してみる
q('''
SELECT sum(amount) AS total,
       count(amount) AS n_not_null,
       count(*) AS n_all,
       sum(amount) / count(amount) AS avg_by_notnull,
       sum(amount) / count(*)      AS avg_by_all
FROM orders
''')

<details>
<summary>何が起きたか</summary>

- `count(*)` は**行数**。12
- `count(amount)` は**NULL でない数**。11
- `count(DISTINCT col)` は種類数。NULL は数えない

**`avg` の分母は `count(col)`、つまり NULL を除いた数。**
`sum / count(*)` とは一致しない。「平均が思ったより高い」の原因はたいていこれ。

`sum` も NULL を無視する。**全部 NULL なら `sum` は `0` ではなく `NULL`**。

```sql
SELECT sum(amount) FROM orders WHERE FALSE;   -- NULL
SELECT count(*)    FROM orders WHERE FALSE;   -- 0
```

`coalesce(sum(amount), 0)` で 0 にしておくと下流が楽になる。

</details>

### 1-4. GROUP BY と NULL

キーが NULL の行はどうなるか。**pandas と挙動が違う。**

**実行する前に、違いを予想する。**

In [ ]:
# A: SQL
q('''
SELECT customer_id, count(*) AS n, sum(amount) AS total
FROM orders
GROUP BY customer_id
ORDER BY customer_id
''')

In [ ]:
# B: pandas (既定)
o = q("SELECT * FROM orders")
o.groupby("customer_id", as_index=False, dropna=True).agg(
    n=("order_id", "size"), total=("amount", "sum"))

<details>
<summary>何が起きたか</summary>

**SQL は NULL を1つのグループとして残す。pandas は既定で捨てる。**

| | NULL キーの行 |
| --- | --- |
| SQL `GROUP BY` | 1グループとして残る |
| pandas `groupby` | **既定で消える**(`dropna=True`) |

SQL のクエリを pandas に移植すると、ここで合計が変わる。
pandas 側で `dropna=False` を付けると揃う。

逆方向に移植するときは、SQL 側で `WHERE customer_id IS NOT NULL` が要る。

</details>

---
## 2. JOIN

### 2-1. INNER と LEFT

結合できなかった行をどうするか。行数を必ず数える。

**実行する前に、違いを予想する。**

In [ ]:
# A: INNER JOIN
q('''
SELECT o.order_id, o.customer_id, c.name, c.tier
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id
''')

In [ ]:
# B: LEFT JOIN
q('''
SELECT o.order_id, o.customer_id, c.name, c.tier
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id
''')

<details>
<summary>何が起きたか</summary>

`A` は **10行**、`B` は **12行**。

`INNER` は両方にある行だけ残す。`O-009`(顧客 `C-99` が customers に無い)と
`O-010`(customer_id が NULL)が**黙って消える**。

`B` の LEFT JOIN なら全部残り、右側が `None` になる。

**「JOIN したら件数が減った」は事故ではなく仕様。** ただし気づかないと事故になる。
JOIN の前後で必ず行数を比べる。

```sql
SELECT count(*) FROM orders;                     -- 12
SELECT count(*) FROM orders o JOIN customers ...;-- 10  ← 2件どこへ行った?
```

`ON` の等値比較でも NULL は一致しないので、
**customer_id が NULL の行は LEFT JOIN でも右側が埋まらない**。

</details>

### 2-2. LEFT JOIN の ON と WHERE

条件を `ON` に書くか `WHERE` に書くか。**結果が変わる。**

**実行する前に、違いを予想する。**

In [ ]:
# A: ON に書く
q('''
SELECT o.order_id, c.name, c.tier
FROM orders o
LEFT JOIN customers c
  ON o.customer_id = c.customer_id AND c.tier = 'gold'
ORDER BY o.order_id
''')

In [ ]:
# B: WHERE に書く
q('''
SELECT o.order_id, c.name, c.tier
FROM orders o
LEFT JOIN customers c
  ON o.customer_id = c.customer_id
WHERE c.tier = 'gold'
ORDER BY o.order_id
''')

<details>
<summary>何が起きたか</summary>

`A` は **12行**(全注文。gold でなければ右が NULL)。
`B` は **6行**。

順番はこう。

```
1. ON の条件で結合する       (LEFT なので左は全部残る)
2. WHERE で絞る             ← ここで右が NULL の行も落ちる
```

`WHERE c.tier = 'gold'` は、右が NULL の行に対して `NULL = 'gold'` → `NULL` を返し、
その行を落とす。**結果として LEFT JOIN が INNER JOIN に化ける。**

規則:

- **右側を絞る条件は `ON` に書く**
- **左側を絞る条件は `WHERE` に書く**

`WHERE c.tier IS NULL` だけは例外で、これは anti join の書き方になる。

</details>

### 2-3. 1対多で行が増える

JOIN で行数が増えるとき、集計が壊れる。

**実行する前に、違いを予想する。**

In [ ]:
# A: そのまま JOIN
q('''
SELECT o.order_id, o.amount, p.payment_id, p.amount AS paid
FROM orders o
JOIN payments p ON o.order_id = p.order_id
ORDER BY o.order_id
''')

In [ ]:
# B: 集計してから JOIN
q('''
SELECT o.order_id, o.amount, p.n_payments, p.paid
FROM orders o
LEFT JOIN (
  SELECT order_id, count(*) AS n_payments, sum(amount) AS paid
  FROM payments GROUP BY order_id
) p ON o.order_id = p.order_id
ORDER BY o.order_id
''')

<details>
<summary>何が起きたか</summary>

`O-001` には支払いが2件あるので、`A` では `O-001` の行が**2行に増える**。
この状態で `sum(o.amount)` すると、**注文金額が二重に計上される**。

```sql
SELECT sum(o.amount) FROM orders o JOIN payments p ON o.order_id = p.order_id;
```

これは「JOIN による水増し(fan-out)」と呼ばれる、集計が合わない原因の筆頭。

対策は `B`。**多側を先に集約して1対1にしてから JOIN する。**

JOIN を書いたら必ず自問する: **この結合は1対1か、1対多か。**
1対多なら、集計をどちらでやるかを決めてから書く。

</details>

### 2-4. ANTI JOIN — 片方にしか無いもの

「注文の無い顧客」「顧客の無い注文」を出す2つの書き方。

**実行する前に、違いを予想する。**

In [ ]:
# A: LEFT JOIN + IS NULL
q('''
SELECT c.customer_id, c.name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
''')

In [ ]:
# B: ANTI JOIN (DuckDB)
q('''
SELECT c.customer_id, c.name
FROM customers c
ANTI JOIN orders o ON c.customer_id = o.customer_id
''')

<details>
<summary>何が起きたか</summary>

どちらも `C-05` の1件。

`A` は移植性が高い(どのDBでも動く)。`B` は DuckDB / Spark などが
持っている専用構文で、意図が明確。

`A` を書くときは **`WHERE 右.列 IS NULL`** の列に注意する。
NULL になりうる列(`o.customer_id` など)を指定すると、
結合できているのに NULL の行まで拾ってしまう。**主キーを指定する**のが安全。

逆向き(「顧客の無い注文」)も同じ形で書ける。
`FULL OUTER JOIN` にすれば両側の孤児を一度に出せる。

</details>

---
## 3. WHERE と HAVING

### 3-1. 絞る位置

集約の前に絞るか、後に絞るか。

**実行する前に、違いを予想する。**

In [ ]:
# A: WHERE (集約の前)
q('''
SELECT region, count(*) AS n, sum(amount) AS total
FROM orders
WHERE status = 'completed'
GROUP BY region
ORDER BY region
''')

In [ ]:
# B: HAVING (集約の後)
q('''
SELECT region, count(*) AS n, sum(amount) AS total
FROM orders
GROUP BY region
HAVING sum(amount) > 5000
ORDER BY region
''')

<details>
<summary>何が起きたか</summary>

実行の順番はこう。

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

- `WHERE` は**行**を絞る。集約の前なので、集約結果に影響する
- `HAVING` は**グループ**を絞る。集約の後なので `sum()` を条件に使える

`WHERE sum(amount) > 5000` は書けない(その時点でまだ集約していない)。

**両方書けるなら `WHERE` に書く。** 先に行を減らしたほうが速い。

`SELECT` が `GROUP BY` より後なので、`SELECT` で付けた別名は
`WHERE` では使えない。`ORDER BY` では使える(それより後だから)。

</details>

---
## 4. 窓関数

**「キーごとに最新の1行」を SQL で書く。** pandas の
`drop_duplicates(keep="last")` に対応する。

In [ ]:
# 訂正が後から届いた状況を作る (pandas-03 と同じ形)
con.execute("""
CREATE OR REPLACE TABLE raw AS
SELECT * FROM (VALUES
  ('O-002', TIMESTAMP '2024-02-01 09:00', 980,  'pending'),
  ('O-001', TIMESTAMP '2024-02-01 10:00', 2400, 'completed'),
  ('O-001', TIMESTAMP '2024-02-02 10:00', NULL, 'cancelled'),
  ('O-003', TIMESTAMP '2024-02-01 11:00', 3600, 'completed')
) AS t(order_id, ingested_at, amount, status);
""")
q("SELECT * FROM raw ORDER BY order_id, ingested_at")

### 4-1. row_number で1行選ぶ

「order_id ごとに ingested_at が最新の1行」。窓関数の定番の使い方。

**実行する前に、違いを予想する。**

In [ ]:
# A: 番号を振ってみる
q('''
SELECT *,
       row_number() OVER (PARTITION BY order_id ORDER BY ingested_at DESC) AS rn
FROM raw
ORDER BY order_id, rn
''')

In [ ]:
# B: 1番だけ残す
q('''
SELECT order_id, ingested_at, amount, status
FROM (
  SELECT *, row_number() OVER (PARTITION BY order_id ORDER BY ingested_at DESC) AS rn
  FROM raw
) WHERE rn = 1
ORDER BY order_id
''')

<details>
<summary>何が起きたか</summary>

`OVER (PARTITION BY ... ORDER BY ...)` は「グループごとに並べて番号を振る」。
`GROUP BY` と違って**行が減らない**のがポイントで、
番号を振ってから `WHERE rn = 1` で絞る。

`O-001` の `amount` は **NULL のまま**。pandas の `groupby().last()` のように
別の行から値を拾ってくることはない。**SQL の窓関数は常に行を選ぶ。**

`WHERE rn = 1` を同じ階層に書けないのは、`SELECT` より `WHERE` が
先に評価されるため(3-1 の実行順)。サブクエリか CTE で1段包む。

</details>

### 4-2. QUALIFY で1段減らす

DuckDB / Snowflake / BigQuery にある構文。窓関数の結果を直接絞れる。

**実行する前に、違いを予想する。**

In [ ]:
# A: サブクエリ
q('''
SELECT order_id, amount, status FROM (
  SELECT *, row_number() OVER (PARTITION BY order_id ORDER BY ingested_at DESC) AS rn
  FROM raw
) WHERE rn = 1 ORDER BY order_id
''')

In [ ]:
# B: QUALIFY
q('''
SELECT order_id, amount, status
FROM raw
QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY ingested_at DESC) = 1
ORDER BY order_id
''')

<details>
<summary>何が起きたか</summary>

同じ結果。`QUALIFY` は「窓関数に対する HAVING」。

```
WHERE   … 行を絞る    (集約の前)
HAVING  … 群を絞る    (集約の後)
QUALIFY … 窓関数の結果で絞る
```

PostgreSQL や MySQL には無いので、移植性が要るなら `A`。
分析用のDWH(DuckDB / Snowflake / BigQuery)ならこちらのほうが読みやすい。

</details>

### 4-3. row_number / rank / dense_rank

同順位の扱い。pandas の `rank(method=)` に対応する。

**実行する前に、違いを予想する。**

In [ ]:
# A: 3つ並べる
q('''
SELECT region, amount,
       row_number() OVER (PARTITION BY region ORDER BY amount DESC) AS rn,
       rank()       OVER (PARTITION BY region ORDER BY amount DESC) AS rk,
       dense_rank() OVER (PARTITION BY region ORDER BY amount DESC) AS drk
FROM orders
WHERE region = 'east'
ORDER BY amount DESC NULLS LAST
''')

In [ ]:
# B: 同順位のあるデータで
q('''
SELECT x,
       row_number() OVER (ORDER BY x) AS rn,
       rank()       OVER (ORDER BY x) AS rk,
       dense_rank() OVER (ORDER BY x) AS drk
FROM (VALUES (10), (20), (20), (30)) AS t(x)
ORDER BY x
''')

<details>
<summary>何が起きたか</summary>

| 関数 | 10, 20, 20, 30 に対して | pandas |
| --- | --- | --- |
| `row_number()` | 1, 2, 3, 4 | `rank(method="first")` |
| `rank()` | 1, 2, 2, **4** | `rank(method="min")` |
| `dense_rank()` | 1, 2, 2, **3** | `rank(method="dense")` |

**`row_number()` は同順位を作らない。** だから「1行だけ選ぶ」用途では
`rank()` ではなく `row_number()` を使う(`rank()` だと同着で2行返る)。

ただし `ORDER BY` が同着だと、`row_number()` が**どちらを1にするかは不定**。
pandas と同じで、**第2キーを足して全順序にする**。

```sql
row_number() OVER (PARTITION BY order_id ORDER BY ingested_at DESC, amount DESC)
```

</details>

### 4-4. ORDER BY と NULL の位置

並べ替えたとき NULL はどこに行くか。DB によって既定が違う。

**実行する前に、違いを予想する。**

In [ ]:
# A: 既定
q('''
SELECT order_id, amount FROM orders
ORDER BY amount DESC
LIMIT 4
''')

In [ ]:
# B: 明示する
q('''
SELECT order_id, amount FROM orders
ORDER BY amount DESC NULLS LAST
LIMIT 4
''')

<details>
<summary>何が起きたか</summary>

DuckDB と PostgreSQL の既定は「`ASC` なら NULL は最後、`DESC` なら最初」。
MySQL は逆。**DB によって違う。**

`NULLS FIRST` / `NULLS LAST` を明示すれば、どこでも同じ結果になる。

`ORDER BY amount DESC` で「最大の行を1つ取る」つもりのクエリは、
NULL が先頭に来る DB では**NULL の行が取れる**。
`LIMIT 1` と組み合わせるときは特に注意する。

</details>

---
## 練習

In [ ]:
# 練習1: 顧客ごとに、注文件数と金額合計を出す。
#        customers に載っていない顧客 (C-99) と、customer_id が欠損の注文も
#        1行として残すこと。列は customer_id, name, n_orders, total。
#        name は customers に無ければ NULL のままでよい。

ans = q("""
    -- ここに書く
    SELECT NULL AS customer_id, NULL AS name, NULL AS n_orders, NULL AS total
""")
display(ans)

assert len(ans) == 6, f"6行のはず: {len(ans)}"
assert ans["n_orders"].sum() == 12, f"注文が全部数えられていない: {ans['n_orders'].sum()}"
assert ans["total"].sum() == 23220
print("OK")

In [ ]:
# 練習2: 支払いが1件も無い注文の order_id を出す。order_id 昇順。

ans = q("""
    -- ここに書く
    SELECT NULL AS order_id
""")
display(ans)

assert ans["order_id"].tolist() == ["O-002", "O-004", "O-006", "O-007",
                                    "O-008", "O-009", "O-010", "O-011",
                                    "O-012"], ans["order_id"].tolist()
print("OK")

In [ ]:
# 練習3: 地域ごとに金額が最も大きい注文を1件ずつ出す。
#        列は region, order_id, amount。region 昇順。
#        (同額のときは order_id の小さいほうを選ぶ)

ans = q("""
    -- ここに書く
    SELECT NULL AS region, NULL AS order_id, NULL AS amount
""")
display(ans)

assert len(ans) == 4, f"4行のはず: {len(ans)}"
assert ans["region"].tolist() == ["east", "north", "south", "west"]
assert ans["order_id"].tolist() == ["O-009", "O-005", "O-007", "O-004"], ans["order_id"].tolist()
print("OK")

---
## まとめ

### NULL

| 書き方 | 意味 |
| --- | --- |
| `= NULL` | **常に NULL。0件になる** |
| `IS NULL` / `IS NOT NULL` | 正しい判定 |
| `IS NOT DISTINCT FROM` | NULL 同士も等しいとみなす比較 |
| `NOT IN (サブクエリ)` | **NULL が1つでもあれば0件** |
| `NOT EXISTS` | 安全。第一候補 |
| `count(*)` / `count(col)` | 行数 / 非NULL数 |
| `avg(col)` | 分母は `count(col)`。**NULL を除く** |
| `sum(col)` | 全部 NULL なら 0 ではなく **NULL** |
| `GROUP BY` | NULL を**1グループとして残す**(pandas と逆) |

### JOIN

| | 意味 |
| --- | --- |
| `INNER` | 両方にある行だけ。**黙って減る** |
| `LEFT` | 左を全部残す |
| `ANTI` / `LEFT + IS NULL` | 片方にしか無いもの |
| 条件を `ON` に書く | 右側を絞る |
| 条件を `WHERE` に書く | **LEFT JOIN が INNER に化ける** |
| 1対多 | **行が増えて二重計上**。先に集約する |

### 窓関数

| | |
| --- | --- |
| `OVER (PARTITION BY ... ORDER BY ...)` | 行を減らさずにグループ内で計算 |
| `row_number()` | 同順位を作らない。**1行選ぶならこれ** |
| `rank()` / `dense_rank()` | 同順位あり。次を飛ばす / 飛ばさない |
| `QUALIFY` | 窓関数の結果で絞る (DuckDB/Snowflake/BigQuery) |
| `NULLS FIRST` / `NULLS LAST` | **DBによって既定が違う。明示する** |

### 実行順

```
FROM → WHERE → GROUP BY → HAVING → 窓関数 → QUALIFY → SELECT → ORDER BY → LIMIT
```

`SELECT` の別名が `WHERE` で使えず `ORDER BY` で使えるのは、この順番のため。